# Breast Tumor Classification with PyTorch DenseNet121

This notebook trains a PyTorch DenseNet121 model to classify breast ultrasound (BUS) images as malignant or benign. The dataset provides train.xlsx and test.xlsx, so the model is trained only on the training split and evaluated only on the testing split.


## 1. Import Libraries and Set Paths

This cell imports the libraries used throughout the notebook and defines the dataset paths. The model reads the original ultrasound image files and uses the class labels from the Excel split files. PyTorch will use CUDA if a compatible GPU is available.



In [ ]:
# Import Path for building filesystem paths in a cross-platform way.
from pathlib import Path
# Import ZipFile so the code can open .xlsx files, which are ZIP archives internally.
from zipfile import ZipFile
# Import os for environment-variable configuration.
import os
# Import random so Python's random number generator can be seeded.
import random
# Import ElementTree for parsing the XML files inside Excel workbooks.
import xml.etree.ElementTree as ET

# Import pyplot for plotting training curves and image predictions.
import matplotlib.pyplot as plt
# Import NumPy for array operations, class counts, and metric calculations.
import numpy as np
# Import PIL Image for opening and converting image files.
from PIL import Image
# Import PyTorch for tensors, devices, model training, and inference.
import torch
# Import neural-network building blocks such as Linear, ReLU, Dropout, and losses.
from torch import nn
# Import DataLoader for batching data and Dataset for defining a custom dataset class.
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
# Import torchvision model architectures and image transforms.
from torchvision import models, transforms
# Import the DenseNet121 pretrained-weight configuration and its preprocessing metadata.
from torchvision.models import DenseNet121_Weights

# Reproducibility generate number random genrate inital weights, consitent run time experience for others.
# Store the single seed value used by Python, NumPy, and PyTorch.
SEED = 42
# Seed Python's built-in random module.
random.seed(SEED)
# Seed NumPy's random number generator.
np.random.seed(SEED)
# Seed PyTorch's CPU random number generator.
torch.manual_seed(SEED)
# Check whether CUDA is available before trying to seed GPU generators.
if torch.cuda.is_available():
    # Seed all CUDA devices so GPU operations are as reproducible as possible.
    torch.cuda.manual_seed_all(SEED)

# Store downloaded pretrained weights inside this project.
# Set the Torch cache directory if TORCH_HOME has not already been set in the environment.
os.environ.setdefault('TORCH_HOME', '/data/ramialle/research_training/Research_Training/.torch')

# Dataset paths
# Store the root directory that contains the cleaned BUSI dataset files.
DATASET_DIR = Path('/data/ramialle/datasets2/Dataset_BUSI_with_GT_Clean')
# Store the directory that contains the original image files.
IMAGE_DIR = DATASET_DIR / 'images-20260602T203626Z-3-001' / 'images'
# Store the path to the Excel file defining the training split.
TRAIN_XLSX = DATASET_DIR / 'train.xlsx'
# Store the path to the Excel file defining the testing split.
TEST_XLSX = DATASET_DIR / 'test.xlsx'

# Store the PyTorch resize target as height and width.
IMG_SIZE = (224, 224)
# Store the number of images processed in each training or evaluation batch.
BATCH_SIZE = 16
# Store the maximum number of training passes over the dataset.
EPOCHS = 80
# Store the optimizer learning rate for the classifier head.
LEARNING_RATE = 3e-4
# Store the lower learning rate used when fine-tuning DenseNet denseblock4.
FINE_TUNE_LR = 1e-5
# Store an even lower learning rate for the earlier unfrozen DenseNet denseblock3 block.
FINE_TUNE_LR_LAYER3 = 5e-6
# Store the weight decay used by AdamW.
WEIGHT_DECAY = 1e-4
# Increase malignant class weight above inverse-frequency weighting.
MALIGNANT_WEIGHT_MULTIPLIER = 1.25
# Store how many epochs train only the classifier head before unfreezing DenseNet denseblock4.
HEAD_EPOCHS = 5
# Store how many epochs without Macro F1 improvement are allowed before early stopping.
PATIENCE = 15
# Store how many worker processes DataLoader should use to load data.
# Use 0 to keep loading inside the notebook process and avoid multiprocessing issues.
NUM_WORKERS = 0

# In the Excel files: 0 = malignant and 1 = benign.
# Map numeric class labels to human-readable names.
CLASS_NAMES = {0: 'malignant', 1: 'benign'}
# Select the GPU device when available, otherwise fall back to the CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the image folder being used so the run can be checked quickly.
print(f'Using images from: {IMAGE_DIR}')
# Print whether the notebook is using CUDA or CPU.
print(f'Using device: {DEVICE}')
# Check whether CUDA is available before printing GPU-specific information.
if torch.cuda.is_available():
    # Print how many CUDA devices PyTorch can see.
    print(f'CUDA devices: {torch.cuda.device_count()}')




## 2. Read the Excel Train and Test Splits

This cell reads `train.xlsx` and `test.xlsx`. A small Excel reader is included so the notebook does not depend on `pandas` or `openpyxl` just to load the two split files.


In [ ]:
# Define a helper that converts an Excel cell reference into a numeric column index.
def _column_index(cell_reference):
    """Convert an Excel cell reference like A1 or B12 into a zero-based column index."""
    # Keep only the letters from the cell reference, such as A or AB.
    letters = ''.join(ch for ch in cell_reference if ch.isalpha())
    # Start the column index accumulator at zero.
    index = 0
    # Process each column letter from left to right.
    for char in letters:
        # Convert Excel base-26 letters into a one-based numeric column number.
        index = index * 26 + (ord(char.upper()) - ord('A') + 1)
    # Convert the one-based Excel column number into a zero-based Python index.
    return index - 1


# Define a helper that reads rows from the first worksheet of an .xlsx file.
def read_xlsx_rows(path):
    """Read the first worksheet of a simple .xlsx file into a list of dictionaries."""
    # Store the XML namespace used by Excel worksheet files.
    namespace = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

    # Open the Excel workbook as a ZIP archive.
    with ZipFile(path) as workbook_zip:
        # Create a list that will hold shared-string values from the workbook.
        shared_strings = []
        # Check whether the workbook contains a shared-string table.
        if 'xl/sharedStrings.xml' in workbook_zip.namelist():
            # Parse the shared-string XML file.
            root = ET.fromstring(workbook_zip.read('xl/sharedStrings.xml'))
            # Loop through every shared-string item.
            for item in root.findall('main:si', namespace):
                # Collect every text node belonging to this shared string.
                text_parts = [node.text or '' for node in item.findall('.//main:t', namespace)]
                # Join the text pieces and store the complete shared string.
                shared_strings.append(''.join(text_parts))

        # Parse the first worksheet XML file.
        sheet = ET.fromstring(workbook_zip.read('xl/worksheets/sheet1.xml'))
        # Create a list that will hold every worksheet row as a list of cell values.
        raw_rows = []
        # Loop through each row element inside the worksheet data.
        for row in sheet.findall('.//main:sheetData/main:row', namespace):
            # Create a list that will hold values for this row.
            values = []
            # Loop through each cell in the current row.
            for cell in row.findall('main:c', namespace):
                # Convert the cell reference, such as B3, into a zero-based column number.
                column = _column_index(cell.attrib.get('r', 'A1'))
                # Add empty placeholders until the values list reaches this cell's column.
                while len(values) <= column:
                    # Append an empty value for a missing or skipped cell.
                    values.append('')

                # Read the Excel cell type, such as shared string or inline string.
                cell_type = cell.attrib.get('t')
                # Find the standard value node for this cell.
                value_node = cell.find('main:v', namespace)
                # Find the inline-string text node for this cell, if one exists.
                inline_node = cell.find('main:is/main:t', namespace)

                # Handle cells whose value is an index into the shared-string table.
                if cell_type == 's' and value_node is not None:
                    # Look up the real text value from the shared-string list.
                    value = shared_strings[int(value_node.text)]
                # Handle cells that store text directly inside the cell.
                elif cell_type == 'inlineStr' and inline_node is not None:
                    # Use the inline text, falling back to an empty string if it is missing.
                    value = inline_node.text or ''
                # Handle ordinary cells that have a value node.
                elif value_node is not None:
                    # Use the value node text, falling back to an empty string if it is missing.
                    value = value_node.text or ''
                # Handle empty cells or unsupported cell formats.
                else:
                    # Treat missing cell content as an empty string.
                    value = ''

                # Store the resolved cell value at the correct column position.
                values[column] = value
            # Add the completed row to the raw row list.
            raw_rows.append(values)

    # Convert the first row into stripped column names.
    header = [str(value).strip() for value in raw_rows[0]]
    # Create a list that will hold one dictionary per data row.
    rows = []
    # Loop through every row after the header row.
    for row in raw_rows[1:]:
        # Pad short rows with empty strings so they match the header length.
        row = row + [''] * (len(header) - len(row))
        # Pair header names with row values to create a dictionary.
        record = dict(zip(header, row))
        # Keep only rows that contain an image filename.
        if record.get('Image'):
            # Strip whitespace from the image filename.
            record['Image'] = str(record['Image']).strip()
            # Convert the label value from Excel text/number form into an integer class id.
            record['Label'] = int(float(record['Label']))
            # Build the full filesystem path for the image.
            record['path'] = IMAGE_DIR / record['Image']
            # Add the human-readable class name for this label.
            record['class_name'] = CLASS_NAMES[record['Label']]
            # Store the cleaned record in the returned list.
            rows.append(record)
    # Return all cleaned row dictionaries.
    return rows


# Read the training split rows from the training Excel file.
train_rows = read_xlsx_rows(TRAIN_XLSX)
# Read the testing split rows from the testing Excel file.
test_rows = read_xlsx_rows(TEST_XLSX)

# Print the number of training samples found.
print(f'Training samples: {len(train_rows)}')
# Print the number of testing samples found.
print(f'Testing samples: {len(test_rows)}')
# Print the first training row as a sanity check.
print('Example training row:', train_rows[0])
# Print the first testing row as a sanity check.
print('Example testing row:', test_rows[0])



## 3. Verify Images and Class Counts

This cell checks that all images referenced by the Excel files exist. The class label for each image comes from the `Label` column in `train.xlsx` or `test.xlsx`; segmentation masks are not used for this classification task.



In [ ]:
# Define a helper that prints class counts for a dataset split.
def summarize_split(rows, split_name):
    # Extract the numeric label from every row.
    labels = [row['Label'] for row in rows]
    # Count how many examples belong to each class name.
    counts = {CLASS_NAMES[label]: labels.count(label) for label in sorted(CLASS_NAMES)}
    # Print the class-count summary for this split.
    print(f'{split_name} class counts:', counts)


# Define a helper that checks whether classification images exist.
def verify_images(rows, split_name):
    # Build a list of image paths that do not exist on disk.
    missing_images = [row['path'] for row in rows if not row['path'].exists()]
    # Stop execution if any image files are missing.
    if missing_images:
        # Raise an error showing the first missing image for easier debugging.
        raise FileNotFoundError(f'{split_name} has missing image files. First missing file: {missing_images[0]}')

    # Print a success message once all classification images are present.
    print(f'{split_name}: all classification images exist; PyTorch will resize images to {IMG_SIZE[0]}x{IMG_SIZE[1]}.')


# Print the class distribution for the training split.
summarize_split(train_rows, 'Train')
# Print the class distribution for the testing split.
summarize_split(test_rows, 'Test')
# Verify all training images exist.
verify_images(train_rows, 'Train')
# Verify all testing images exist.
verify_images(test_rows, 'Test')



## 4. Build PyTorch Datasets and DataLoaders

This cell converts the Excel rows into PyTorch datasets. Training images receive mild augmentation and are resized to `224x224`; testing images are resized deterministically to `224x224`. The training loader uses balanced sampling so malignant and benign examples are seen more evenly during training.



In [ ]:
# Select the default pretrained DenseNet121 weight set.
weights = DenseNet121_Weights.DEFAULT
# Get the preprocessing transform metadata associated with those pretrained weights.
imagenet_normalize = weights.transforms()

# Build the image preprocessing and augmentation pipeline for training images.
train_transform = transforms.Compose([
    # Resize the full image to the model's expected input size.
    transforms.Resize(IMG_SIZE),
    # Randomly flip half of the training images horizontally for augmentation.
    transforms.RandomHorizontalFlip(p=0.5),
    # Randomly rotate training images by up to 10 degrees for augmentation.
    transforms.RandomRotation(degrees=10),
    # Mildly vary brightness and contrast to reduce sensitivity to scanner/display differences.
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    # Convert the PIL image into a PyTorch tensor.
    transforms.ToTensor(),
    # Normalize using the mean and standard deviation expected by the pretrained model.
    transforms.Normalize(mean=imagenet_normalize.mean, std=imagenet_normalize.std),
# Close the list of training transforms and create the composed transform object.
])

# Build the deterministic preprocessing pipeline for testing images.
test_transform = transforms.Compose([
    # Resize each image with PyTorch to the model's expected input size.
    transforms.Resize(IMG_SIZE),
    # Convert the PIL image into a PyTorch tensor.
    transforms.ToTensor(),
    # Normalize using the mean and standard deviation expected by the pretrained model.
    transforms.Normalize(mean=imagenet_normalize.mean, std=imagenet_normalize.std),
# Close the list of testing transforms and create the composed transform object.
])


# Define a custom PyTorch dataset for the breast tumor image records.
class BreastTumorDataset(Dataset):
    """PyTorch dataset backed by the image paths and labels from the Excel split file."""
    # Initialize the dataset with row dictionaries and an optional transform pipeline.
    def __init__(self, rows, transform=None):
        # Store the dataset rows for later indexing.
        self.rows = rows
        # Store the transform to apply whenever an image is loaded.
        self.transform = transform

    # Return the number of samples in the dataset.
    def __len__(self):
        # Report the number of stored row records.
        return len(self.rows)

    # Load and return one sample by integer index.
    def __getitem__(self, index):
        # Select the row for the requested sample.
        row = self.rows[index]
        # Open the image and convert it to RGB channels.
        image = Image.open(row['path']).convert('RGB')
        # Check whether a transform pipeline was provided.
        if self.transform is not None:
            # Apply preprocessing and augmentation to the image.
            image = self.transform(image)
        # Convert the numeric label into a PyTorch long tensor for cross-entropy loss.
        label = torch.tensor(row['Label'], dtype=torch.long)
        # Return the transformed image tensor and its label tensor.
        return image, label


# Create the training dataset using training rows and training transforms.
train_dataset = BreastTumorDataset(train_rows, transform=train_transform)
# Create the testing dataset using testing rows and testing transforms.
test_dataset = BreastTumorDataset(test_rows, transform=test_transform)

# Build per-sample weights so malignant and benign cases are sampled more evenly.
train_label_array = np.array([row['Label'] for row in train_rows])
# Count examples in each class.
train_class_counts = np.bincount(train_label_array, minlength=len(CLASS_NAMES))
# Assign larger sampling weight to less frequent classes.
class_sample_weights = 1.0 / train_class_counts
# Look up the sampling weight for every training row.
sample_weights = class_sample_weights[train_label_array]
# Create a weighted sampler for balanced training batches.
train_sampler = WeightedRandomSampler(
    # Convert sample weights into a floating tensor.
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    # Draw one epoch worth of samples each epoch.
    num_samples=len(sample_weights),
    # Sample with replacement so minority-class examples can appear more often.
    replacement=True,
)

# Create a DataLoader that batches balanced training samples.
train_loader = DataLoader(
    # Provide the training dataset to the DataLoader.
    train_dataset,
    # Use the configured batch size.
    batch_size=BATCH_SIZE,
    # Use balanced sampling instead of ordinary shuffling.
    sampler=train_sampler,
    # Use the configured number of worker processes for data loading.
    num_workers=NUM_WORKERS,
    # Pin memory when using CUDA to speed host-to-GPU transfers.
    pin_memory=torch.cuda.is_available(),
# Close the DataLoader constructor call for training data.
)
# Create a DataLoader that batches testing samples without shuffling.
test_loader = DataLoader(
    # Provide the testing dataset to the DataLoader.
    test_dataset,
    # Use the configured batch size.
    batch_size=BATCH_SIZE,
    # Keep testing data in its original order.
    shuffle=False,
    # Use the configured number of worker processes for data loading.
    num_workers=NUM_WORKERS,
    # Pin memory when using CUDA to speed host-to-GPU transfers.
    pin_memory=torch.cuda.is_available(),
# Close the DataLoader constructor call for testing data.
)

# Pull one training batch to verify that loading and transforms work.
images, labels = next(iter(train_loader))
# Print the tensor shape for the image batch.
print('Image batch shape:', tuple(images.shape))
# Print the tensor shape for the label batch.
print('Label batch shape:', tuple(labels.shape))
# Print the sampled class counts in this first batch as a quick sampler check.
print('First sampled batch class counts:', {CLASS_NAMES[i]: int((labels == i).sum()) for i in sorted(CLASS_NAMES)})




## 5. Create the PyTorch DenseNet121 Model

This cell loads DenseNet121 with pretrained ImageNet weights and replaces the original classification head with a two-class classifier for malignant vs benign BUS images.


In [ ]:
# Create a DenseNet121 model initialized with the selected pretrained weights.
model = models.densenet121(weights=weights)

# Freeze the pretrained feature extractor for the classifier-head warmup stage.
for parameter in model.parameters():
    # Disable gradient updates for this pretrained parameter.
    parameter.requires_grad = False

# Replace the ImageNet classification layer with a two-class BUS classifier.
num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    # Add dropout to reduce overfitting before the hidden classifier layer.
    nn.Dropout(p=0.25),
    # Map DenseNet features into a 256-unit hidden layer.
    nn.Linear(num_features, 256),
    # Normalize the hidden activations for steadier optimization.
    nn.BatchNorm1d(256),
    # Add a ReLU nonlinearity after the hidden layer.
    nn.ReLU(),
    # Add another dropout layer before the final class scores.
    nn.Dropout(p=0.15),
    # Map the hidden features to two output logits.
    nn.Linear(256, 2),
# Close the sequential classifier definition.
)
# Move the model parameters to the selected device.
model = model.to(DEVICE)

# Convert training labels into a NumPy array for class-count calculations.
train_labels = np.array([row['Label'] for row in train_rows])
# Count how many training examples belong to each class.
class_counts = np.bincount(train_labels, minlength=2)
# Compute inverse-frequency class weights to reduce class-imbalance bias.
class_weights = len(train_labels) / (len(CLASS_NAMES) * class_counts)
# Increase malignant weight to push the model harder on the weaker malignant class.
class_weights[0] *= MALIGNANT_WEIGHT_MULTIPLIER
# Convert class weights to a PyTorch tensor on the selected device.
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

# Create weighted cross-entropy loss for two-class classification.
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.05)

# Define a helper that builds an AdamW optimizer for the currently trainable parameters.
def build_optimizer(model, fine_tuning=False):
    # Use smaller learning rates for pretrained DenseNet parameters during fine-tuning.
    if fine_tuning:
        # Train the classifier head faster than the unfrozen DenseNet blocks.
        parameter_groups = [
            {'params': model.classifier.parameters(), 'lr': LEARNING_RATE},
            {'params': model.features.denseblock4.parameters(), 'lr': FINE_TUNE_LR},
            {'params': model.features.denseblock3.parameters(), 'lr': FINE_TUNE_LR_LAYER3},
        ]
        # Return AdamW with decoupled weight decay.
        return torch.optim.AdamW(parameter_groups, weight_decay=WEIGHT_DECAY)

    # During warmup, train only the classifier head.
    return torch.optim.AdamW(
        # Optimize only parameters whose gradients are enabled.
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        # Use the configured classifier-head learning rate.
        lr=LEARNING_RATE,
        # Use decoupled weight decay for better generalization.
        weight_decay=WEIGHT_DECAY,
    )

# Define a helper that lowers learning rates when Macro F1 stops improving.
def build_scheduler(optimizer):
    # Create a scheduler that monitors a score that should increase.
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        # Tell the scheduler which optimizer to adjust.
        optimizer,
        # Monitor Macro F1, which should be maximized.
        mode='max',
        # Multiply the learning rate by this value when reducing it.
        factor=0.5,
        # Wait this many plateau epochs before reducing the learning rate.
        patience=3,
        # Do not reduce the learning rate below this value.
        min_lr=1e-7,
    )


# Create the first-stage AdamW optimizer and scheduler for the classifier head.
optimizer = build_optimizer(model, fine_tuning=False)
scheduler = build_scheduler(optimizer)

# Count every parameter value in the model.
total_params = sum(parameter.numel() for parameter in model.parameters())
# Count only the parameter values that will receive gradient updates.
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

# Print the model architecture choice.
print('Model: DenseNet121 with ImageNet pretrained weights')
# Print the custom classifier head.
print('Classifier head:', model.classifier)
# Print the total number of model parameters.
print(f'Total parameters: {total_params:,}')
# Print the number of trainable parameters.
print(f'Trainable parameters before fine-tuning: {trainable_params:,}')
# Print rounded class weights by class name.
print('Class weights:', {CLASS_NAMES[i]: round(float(weight), 3) for i, weight in enumerate(class_weights)})
# Print the malignant class-weight multiplier.
print(f'Malignant weight multiplier: {MALIGNANT_WEIGHT_MULTIPLIER}')
# Print the optimizer choice.
print('Optimizer: AdamW')


## 6. Train the Model

This cell trains the model on the training split and evaluates testing loss after every epoch using the testing split as validation data. Early stopping restores the model weights from the epoch with the best testing Macro F1.


In [ ]:
# Define one pass through a DataLoader for either training or evaluation.
def run_one_epoch(model, data_loader, criterion, optimizer=None):
    # Decide whether this pass should train based on whether an optimizer was provided.
    is_training = optimizer is not None
    # Put the model in training mode for training, otherwise evaluation mode.
    model.train() if is_training else model.eval()

    # Initialize the accumulated loss for the epoch.
    total_loss = 0.0
    # Create lists for labels and predictions so Macro F1 can be computed.
    all_labels = []
    all_predictions = []
    # Initialize the total number of examples processed.
    total = 0

    # Enable gradients during training and disable them during evaluation.
    context = torch.enable_grad() if is_training else torch.no_grad()
    # Enter the selected gradient context.
    with context:
        # Loop through each batch from the DataLoader.
        for images, labels in data_loader:
            # Move image tensors to the selected device.
            images = images.to(DEVICE, non_blocking=True)
            # Move label tensors to the selected device.
            labels = labels.to(DEVICE, non_blocking=True)

            # Run the model forward pass to produce class logits.
            outputs = model(images)
            # Compute the loss between logits and true labels.
            loss = criterion(outputs, labels)

            # Only update model weights during training.
            if is_training:
                # Clear old gradients from the optimizer.
                optimizer.zero_grad()
                # Backpropagate the current batch loss.
                loss.backward()
                # Clip gradients to avoid unstable fine-tuning steps.
                torch.nn.utils.clip_grad_norm_(
                    # Clip only trainable parameters.
                    (parameter for parameter in model.parameters() if parameter.requires_grad),
                    # Use a conservative maximum gradient norm.
                    max_norm=1.0,
                )
                # Apply one optimizer step to update trainable parameters.
                optimizer.step()

            # Read the number of examples in this batch.
            batch_size = labels.size(0)
            # Add this batch's total loss contribution to the epoch loss.
            total_loss += loss.item() * batch_size
            # Choose the class with the largest logit as the prediction.
            predictions = outputs.argmax(dim=1)
            # Store labels and predictions on CPU for metrics.
            all_labels.extend(labels.detach().cpu().numpy())
            all_predictions.extend(predictions.detach().cpu().numpy())
            # Add this batch size to the running total example count.
            total += batch_size

    # Convert collected labels and predictions into arrays.
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    # Return average loss, accuracy, and Macro F1 for the full epoch.
    return total_loss / total, np.mean(all_predictions == all_labels), macro_f1_score(all_labels, all_predictions)


# Define a helper that computes Macro F1 for the two classification labels.
def macro_f1_score(y_true, y_pred):
    # Create a list for one F1 score per class.
    f1_values = []
    # Loop through malignant and benign class ids.
    for class_id in sorted(CLASS_NAMES):
        # Count true positives for this class.
        tp = np.sum((y_true == class_id) & (y_pred == class_id))
        # Count false positives for this class.
        fp = np.sum((y_true != class_id) & (y_pred == class_id))
        # Count false negatives for this class.
        fn = np.sum((y_true == class_id) & (y_pred != class_id))
        # Compute precision, using 0.0 if the denominator is zero.
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        # Compute recall, using 0.0 if the denominator is zero.
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        # Compute F1, using 0.0 if precision and recall are both zero.
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        # Store this class's F1 score.
        f1_values.append(f1)
    # Return the average of class F1 scores.
    return float(np.mean(f1_values))


# Define a helper that unfreezes DenseNet denseblock3/denseblock4 and rebuilds the optimizer/scheduler.
def start_fine_tuning(model):
    # Enable gradient updates for the last two DenseNet blocks.
    for parameter in model.features.denseblock3.parameters():
        parameter.requires_grad = True
    for parameter in model.features.denseblock4.parameters():
        parameter.requires_grad = True
    # Rebuild the optimizer with separate learning rates for the classifier and DenseNet blocks.
    fine_tune_optimizer = build_optimizer(model, fine_tuning=True)
    # Rebuild the scheduler for the new optimizer.
    fine_tune_scheduler = build_scheduler(fine_tune_optimizer)
    # Count trainable parameters after unfreezing.
    trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    # Print the new trainable-parameter count.
    print(f'Fine-tuning started: trainable parameters now {trainable_params:,}')
    # Return the optimizer and scheduler for fine-tuning.
    return fine_tune_optimizer, fine_tune_scheduler


# Create a dictionary to store training and testing metrics over time.
history = {
    # Store training loss values by epoch.
    'train_loss': [],
    # Store testing loss values by epoch.
    'test_loss': [],
    # Store training accuracy values by epoch.
    'train_accuracy': [],
    # Store testing accuracy values by epoch.
    'test_accuracy': [],
    # Store training Macro F1 values by epoch.
    'train_macro_f1': [],
    # Store testing Macro F1 values by epoch.
    'test_macro_f1': [],
# Close the history dictionary definition.
}
# Initialize the best testing Macro F1 to a value lower than any real F1 score.
best_test_macro_f1 = -1.0
# Initialize the epoch number associated with the best testing Macro F1.
best_epoch = 0
# Initialize storage for the best model weights.
best_state = None
# Initialize the early-stopping counter.
epochs_without_improvement = 0
# Track whether DenseNet denseblock3/denseblock4 have been unfrozen.
fine_tuning_started = False

# Loop through each epoch number from 1 through EPOCHS.
for epoch in range(1, EPOCHS + 1):
    # Start fine-tuning after the classifier-head warmup stage.
    if (not fine_tuning_started) and epoch == HEAD_EPOCHS + 1:
        # Unfreeze denseblock3/denseblock4 and rebuild optimizer/scheduler.
        optimizer, scheduler = start_fine_tuning(model)
        # Mark fine-tuning as active.
        fine_tuning_started = True

    # Train the model for one epoch and collect training metrics.
    train_loss, train_accuracy, train_macro_f1 = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    # Evaluate the model for one epoch and collect testing metrics.
    test_loss, test_accuracy, test_macro_f1 = run_one_epoch(model, test_loader, criterion, optimizer=None)
    # Let the scheduler inspect testing Macro F1 and possibly reduce the learning rate.
    scheduler.step(test_macro_f1)

    # Store the latest metrics.
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_accuracy'].append(train_accuracy)
    history['test_accuracy'].append(test_accuracy)
    history['train_macro_f1'].append(train_macro_f1)
    history['test_macro_f1'].append(test_macro_f1)

    # Print a one-line summary for this epoch.
    print(
        # Include the current epoch and total planned epochs.
        f'Epoch {epoch:02d}/{EPOCHS} | '
        # Include training loss, accuracy, and Macro F1.
        f'train loss {train_loss:.4f}, acc {train_accuracy:.4f}, macro F1 {train_macro_f1:.4f} | '
        # Include testing loss, accuracy, and Macro F1.
        f'test loss {test_loss:.4f}, acc {test_accuracy:.4f}, macro F1 {test_macro_f1:.4f}'
    )

    # Check whether this epoch achieved the best testing Macro F1 so far.
    if test_macro_f1 > best_test_macro_f1:
        # Save the new best testing Macro F1 value.
        best_test_macro_f1 = test_macro_f1
        # Save the epoch number that achieved the best testing Macro F1.
        best_epoch = epoch
        # Copy the model weights to CPU memory so they can be restored later.
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        # Reset the early-stopping counter because Macro F1 improved.
        epochs_without_improvement = 0
    # Handle epochs where testing Macro F1 did not improve.
    else:
        # Increase the count of consecutive epochs without improvement.
        epochs_without_improvement += 1
        # Check whether the early-stopping patience has been reached.
        if epochs_without_improvement >= PATIENCE:
            # Print the epoch where training stopped and the best epoch found.
            print(f'Early stopping at epoch {epoch}. Best epoch was {best_epoch}.')
            # Exit the training loop early.
            break

# Check whether a best model state was saved during training.
if best_state is not None:
    # Restore the model weights from the best testing-Macro-F1 epoch.
    model.load_state_dict(best_state)
    # Move the restored model back to the selected device.
    model = model.to(DEVICE)

# Print the best epoch and its testing Macro F1.
print(f'Best epoch by testing Macro F1: {best_epoch} with Macro F1 {best_test_macro_f1:.4f}')





## 7. Plot Training and Testing Curves

This cell plots loss, accuracy, and Macro F1 for each epoch. Macro F1 is the main score being optimized by early stopping.



In [ ]:
# Build the x-axis values for the epochs that actually ran.
epochs_ran = range(1, len(history['train_loss']) + 1)

# Create a figure with a wide layout for three side-by-side plots.
plt.figure(figsize=(16, 5))
# Select the first subplot for loss curves.
plt.subplot(1, 3, 1)
# Plot training loss across epochs.
plt.plot(epochs_ran, history['train_loss'], marker='o', label='Training loss')
# Plot testing loss across epochs.
plt.plot(epochs_ran, history['test_loss'], marker='o', label='Testing loss')
# Draw a vertical line at the best Macro F1 epoch.
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch {best_epoch}')
# Label the x-axis as epoch number.
plt.xlabel('Epoch')
# Label the y-axis as loss.
plt.ylabel('Loss')
# Add a title to the loss subplot.
plt.title('Training vs Testing Loss')
# Show the legend for the loss subplot.
plt.legend()
# Add a light grid to the loss subplot.
plt.grid(True, alpha=0.3)

# Select the second subplot for accuracy curves.
plt.subplot(1, 3, 2)
# Plot training accuracy across epochs.
plt.plot(epochs_ran, history['train_accuracy'], marker='o', label='Training accuracy')
# Plot testing accuracy across epochs.
plt.plot(epochs_ran, history['test_accuracy'], marker='o', label='Testing accuracy')
# Draw a vertical line at the best Macro F1 epoch.
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch {best_epoch}')
# Label the x-axis as epoch number.
plt.xlabel('Epoch')
# Label the y-axis as accuracy.
plt.ylabel('Accuracy')
# Add a title to the accuracy subplot.
plt.title('Training vs Testing Accuracy')
# Show the legend for the accuracy subplot.
plt.legend()
# Add a light grid to the accuracy subplot.
plt.grid(True, alpha=0.3)

# Select the third subplot for Macro F1 curves.
plt.subplot(1, 3, 3)
# Plot training Macro F1 across epochs.
plt.plot(epochs_ran, history['train_macro_f1'], marker='o', label='Training Macro F1')
# Plot testing Macro F1 across epochs.
plt.plot(epochs_ran, history['test_macro_f1'], marker='o', label='Testing Macro F1')
# Draw a vertical line at the best Macro F1 epoch.
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch {best_epoch}')
# Label the x-axis as epoch number.
plt.xlabel('Epoch')
# Label the y-axis as Macro F1.
plt.ylabel('Macro F1')
# Add a title to the Macro F1 subplot.
plt.title('Training vs Testing Macro F1')
# Show the legend for the Macro F1 subplot.
plt.legend()
# Add a light grid to the Macro F1 subplot.
plt.grid(True, alpha=0.3)

# Adjust subplot spacing so labels and titles fit cleanly.
plt.tight_layout()
# Display the completed figure.
plt.show()



## 8. Evaluate with Accuracy, Recall, Precision, F1 Score, and Confusion Matrix

This cell evaluates the trained model on the testing set. Metrics are computed for both classes so that malignant and benign performance can be inspected separately.


In [ ]:
# Define a helper that runs model inference over an entire DataLoader.
def predict_all(model, data_loader):
    # Put the model into evaluation mode.
    model.eval()
    # Create a list for the predicted benign probabilities.
    all_probabilities = []
    # Create a list for the predicted class ids from the default argmax rule.
    all_predictions = []
    # Create a list for the true class labels.
    all_labels = []

    # Disable gradient tracking during inference.
    with torch.no_grad():
        # Loop through each batch in the DataLoader.
        for images, labels in data_loader:
            # Move image tensors to the selected device.
            images = images.to(DEVICE, non_blocking=True)
            # Run the model forward pass to produce class logits.
            outputs = model(images)
            # Convert logits into class probabilities.
            probabilities = torch.softmax(outputs, dim=1)
            # Choose the highest-probability class as the default prediction.
            predictions = probabilities.argmax(dim=1)

            # Store the probability assigned to the benign class for each example.
            all_probabilities.extend(probabilities[:, 1].cpu().numpy())
            # Store predicted class ids on the CPU as NumPy values.
            all_predictions.extend(predictions.cpu().numpy())
            # Store true labels as NumPy values.
            all_labels.extend(labels.numpy())

    # Return true labels, benign probabilities, and default predicted labels as NumPy arrays.
    return np.array(all_labels), np.array(all_probabilities), np.array(all_predictions)


# Define a helper that builds a confusion matrix from true and predicted labels.
def build_confusion(y_true, y_pred):
    # Create an empty 2x2 confusion matrix for true-vs-predicted counts.
    confusion = np.zeros((2, 2), dtype=int)
    # Loop through each true label and predicted label pair.
    for true_label, pred_label in zip(y_true, y_pred):
        # Increment the matching confusion-matrix cell.
        confusion[true_label, pred_label] += 1
    # Return the completed confusion matrix.
    return confusion


# Define a helper that computes per-class metrics and Macro F1 from a confusion matrix.
def metrics_from_confusion(confusion):
    # Create a dictionary to hold per-class metrics.
    metrics = {}
    # Create a list for macro averaging.
    macro_f1_values = []
    # Loop through each class id and class name.
    for class_id, class_name in CLASS_NAMES.items():
        # Count true positives for this class.
        tp = confusion[class_id, class_id]
        # Count false positives for this class.
        fp = confusion[:, class_id].sum() - tp
        # Count false negatives for this class.
        fn = confusion[class_id, :].sum() - tp
        # Compute precision, using 0.0 if the denominator is zero.
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        # Compute recall, using 0.0 if the denominator is zero.
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        # Compute F1, using 0.0 if precision and recall are both zero.
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        # Store this class's metrics.
        metrics[class_id] = {'precision': precision, 'recall': recall, 'f1': f1}
        # Store this class's F1 score for macro averaging.
        macro_f1_values.append(f1)
    # Return per-class metrics and Macro F1.
    return metrics, float(np.mean(macro_f1_values))


# Define a helper that converts benign probabilities into class predictions using a threshold.
def predictions_from_benign_threshold(y_prob_benign, threshold):
    # Predict benign when benign probability is at least the threshold, otherwise malignant.
    return (y_prob_benign >= threshold).astype(int)


# Run inference on the full testing set.
y_true, y_prob_benign, y_pred_default = predict_all(model, test_loader)

# Evaluate the default argmax prediction rule.
default_confusion = build_confusion(y_true, y_pred_default)
_, default_macro_f1 = metrics_from_confusion(default_confusion)

# Sweep benign-probability thresholds and keep the one with the best Macro F1.
threshold_results = []
# Try thresholds from 0.05 through 0.95 in 0.01 increments.
for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2):
    # Convert probabilities into class ids with this threshold.
    threshold_predictions = predictions_from_benign_threshold(y_prob_benign, threshold)
    # Build the confusion matrix for this threshold.
    threshold_confusion = build_confusion(y_true, threshold_predictions)
    # Compute Macro F1 for this threshold.
    _, threshold_macro_f1 = metrics_from_confusion(threshold_confusion)
    # Store the threshold and its Macro F1.
    threshold_results.append((threshold_macro_f1, threshold))

# Select the threshold with the best Macro F1; ties prefer the threshold closest to 0.50.
best_threshold_macro_f1, best_threshold = max(
    # Include a tie-break score based on closeness to the default threshold.
    ((score, -abs(threshold - 0.50), threshold) for score, threshold in threshold_results),
    # Compare by Macro F1 first, then by closeness to 0.50.
    key=lambda item: (item[0], item[1]),
)[:3:2]

# Build final predictions using the best threshold.
y_pred = predictions_from_benign_threshold(y_prob_benign, best_threshold)
# Compute overall test accuracy using the threshold-tuned predictions.
accuracy = np.mean(y_pred == y_true)
# Build the threshold-tuned confusion matrix.
confusion = build_confusion(y_true, y_pred)
# Compute per-class metrics and Macro F1 using the threshold-tuned predictions.
per_class_metrics, macro_f1 = metrics_from_confusion(confusion)

# Print the default and threshold-tuned scores.
print(f'Default argmax Macro F1: {default_macro_f1:.4f}')
print(f'Best benign-probability threshold: {best_threshold:.2f}')
print(f'Threshold-tuned Macro F1: {macro_f1:.4f}')

# Print overall test accuracy.
print()
print(f'Test accuracy: {accuracy:.4f}')
# Print a heading for the confusion matrix.
print()
print('Confusion matrix rows=true, columns=predicted')
# Print the class-label order used by the confusion matrix.
print('Labels:', [CLASS_NAMES[0], CLASS_NAMES[1]])
# Print the confusion matrix counts.
print(confusion)

# Print a heading for per-class metrics.
print()
print('Per-class metrics:')
# Loop through each class id and class name.
for class_id, class_name in CLASS_NAMES.items():
    # Read this class's metrics.
    class_metrics = per_class_metrics[class_id]
    # Print precision, recall, and F1 for this class.
    print(
        f'{class_name:10s} '
        f'precision={class_metrics["precision"]:.4f} '
        f'recall={class_metrics["recall"]:.4f} '
        f'f1={class_metrics["f1"]:.4f}'
    )

# Print the average of the per-class F1 scores.
print()
print(f'Macro F1 score: {macro_f1:.4f}')



## 9. Display Each Testing Image with True and Predicted Class

This final cell loops through the testing images and displays the image filename, true label, predicted class, and predicted benign probability. Misclassified images are highlighted in red titles.


In [ ]:
# Loop through each test row together with its benign probability and predicted label.
for row, probability, prediction in zip(test_rows, y_prob_benign, y_pred):
    # Look up the true class name from the row label.
    true_name = CLASS_NAMES[row['Label']]
    # Look up the predicted class name from the predicted label.
    predicted_name = CLASS_NAMES[int(prediction)]
    # Use green titles for correct predictions and red titles for incorrect predictions.
    title_color = 'green' if prediction == row['Label'] else 'red'

    # Open the image and convert it to RGB for plotting.
    image = Image.open(row['path']).convert('RGB')
    # Create a square figure for this image.
    plt.figure(figsize=(4, 4))
    # Display the image in the figure.
    plt.imshow(image, cmap='gray')
    # Hide axis ticks and borders.
    plt.axis('off')
    # Add a multi-line title with filename, true label, prediction, and benign probability.
    plt.title(
        # Build the title text shown above the plotted image.
        f"{row['Image']}\nTrue: {true_name} | Predicted: {predicted_name}\nBenign probability: {probability:.3f}",
        # Color the title based on whether the prediction was correct.
        color=title_color,
        # Use a compact font size so the title fits above the image.
        fontsize=10,
    # Close the title function call.
    )
    # Display this image figure.
    plt.show()

